# **Sesión 31 — Introducción práctica al Machine Learning**
### **¿Cómo podemos pasar de reglas fijas a modelos que aprenden?**

**Data Science UC — Facultad de Matemáticas**

**No necesitas saber programar.** Cada celda de código está comentada y explicada en el texto de arriba. Lo importante es entender **qué** hace cada modelo y **por qué**. Dedicar tiempo a memorizar la sintaxis es tiempo perdido. Práctica en modo prueba y comprueba tus resultados.

---

# **Recordemos las 3 preguntas con las que cerramos la Sesión 30:**

1. ¿Qué variable de su negocio actual es más difícil de medir con exactitud?
2. ¿Dónde confía más en los datos hoy en comparación a hace un año?
3. ¿Qué decisión toma su empresa hoy basándose en reglas fijas?

Hoy vamos a construir, paso a paso, las herramientas para responder la pregunta 3 con un ejemplo práctico.

## Preparación del Entorno
Importamos las librerías que usaremos durante toda la clase. No hace falta instalar nada. Colab ya las trae.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    mean_squared_error, r2_score
)

np.random.seed(42)  # para que todos obtengamos los mismos resultados en clase
plt.rcParams['figure.figsize'] = (7, 4.5)

print('Entorno listo. Empecemos.')

## Sección 1 — Caso de negocio: "Club Activo"

Para que el ejemplo sirva sin importar tu rubro, usaremos un caso genérico de **suscripción / membresía.**

Un cliente paga mensualmente por un servicio (puede ser un gimnasio, un software, un seguro, una editorial, un plan de telefonía — el mecanismo es el mismo).

Generamos datos **sintéticos** (inventados con reglas realistas) para 800 clientes, con estas variables:

| Variable | Significado | Tipo |
|---|---|---|
| `edad` | Edad del cliente | Numérica |
| `antiguedad_meses` | Meses desde que se suscribió | Numérica |
| `uso_mensual` | Veces que usó el servicio en el último mes | Numérica |
| `quejas_soporte` | Número de reclamos en los últimos 3 meses | Numérica |
| `gasto_mensual` | Cuánto paga al mes (CLP miles) | Numérica |
| `cancelo` | 1 si canceló el servicio, 0 si sigue activo | Categórica (objetivo de clasificación) |

Esta tabla es equivalente a un CRM (Customer Relationship Management), un sistema de historia clínica de pacientes, una base de pólizas de seguros, o cualquier registro de clientes/usuarios que ya tengas en tu organización.

In [ ]:
n = 800

edad = np.random.normal(38, 12, n).clip(18, 75)
antiguedad_meses = np.random.exponential(14, n).clip(1, 60)
uso_mensual = np.random.poisson(8, n).clip(0, 30)
quejas_soporte = np.random.poisson(0.6, n).clip(0, 8)

# El gasto depende (con ruido) de la antigüedad y el uso -> relación real que la regresión deberá aprender
gasto_mensual = (25 + 0.9 * antiguedad_meses + 3.2 * uso_mensual
                  + np.random.normal(0, 12, n)).clip(15, None)

# La probabilidad de cancelar sube con las quejas y baja con el uso y la antigüedad
logit = (-0.7 + 1.35 * quejas_soporte - 0.14 * uso_mensual - 0.05 * antiguedad_meses
         + np.random.normal(0, 0.5, n))
prob_cancela = 1 / (1 + np.exp(-logit))
cancelo = (np.random.rand(n) < prob_cancela).astype(int)

df = pd.DataFrame({
    'edad': edad.round(1),
    'antiguedad_meses': antiguedad_meses.round(1),
    'uso_mensual': uso_mensual,
    'quejas_soporte': quejas_soporte,
    'gasto_mensual': gasto_mensual.round(0),
    'cancelo': cancelo
})

# Simulamos algunos valores faltantes, como en cualquier base de datos real (como hicimos en la Sesión 30)
for col in ['edad', 'antiguedad_meses']:
    idx_faltantes = df.sample(frac=0.03, random_state=1).index
    df.loc[idx_faltantes, col] = np.nan

print(f'Tasa de cancelación en la base: {df.cancelo.mean():.1%}')
df.head(8)

> **¿Por qué "inventamos" los datos con una fórmula exacta?**
>
> Estos datos son sintéticos a propósito: escribimos nosotros mismos las reglas que generan `gasto_mensual` y `cancelo`. Lo hacemos así **solo por hoy**, con el objetivo de comprobar más adelante si el modelo fue capaz de "redescubrir" esas reglas mirando únicamente los datos, sin que nadie se las dijera. En un caso real de tu empresa **nunca vas a conocer esas fórmulas de antemano** — si las conocieras, no necesitarías Machine Learning, bastaría con una calculadora.
>
> **El caso de `gasto_mensual` es directo:** una suma ponderada de variables más ruido aleatorio.
>
> **El caso de `cancelo` tiene un paso extra — el logit y la función sigmoide:**
> 1. Primero calculamos un **"puntaje de riesgo" oculto (logit)**: `-0.7 + 1.35×quejas - 0.14×uso - 0.05×antigüedad + ruido`. Este puntaje puede ser cualquier número — negativo, positivo, grande o chico — y por sí solo no es una probabilidad.
> 2. Ese puntaje se transforma en una **probabilidad entre 0 y 1** con la función sigmoide: `1 / (1 + e^(-logit))`. Es el mismo mecanismo matemático que usa la Regresión Logística para convertir un puntaje en una probabilidad — por eso el modelo que entrenaremos más adelante se llama justamente "logístico".
> 3. Con esa probabilidad, simulamos una **moneda cargada por cliente** (`np.random.rand(n) < prob_cancela`) para decidir si cancela o no. Como en la vida real, tenemos así dos clientes con el mismo perfil de riesgo que no necesariamente siempre, terminan tomando la misma decisión.
>
> De aquí en adelante actuamos como si no supiéramos ninguna de las dos fórmulas: se las escondemos a los modelos, los entrenamos, y solo al final compararemos lo que aprendieron contra la regla original.

## Sección 2 — Recordemos de la Sesión 30: imputación

Antes de entrenar cualquier modelo, aplicamos la **regla de decisión de imputación** vista la sesión pasada:
- `edad` → distribución razonablemente simétrica, sin atípicos extremos → imputamos con la **media**.
- `antiguedad_meses` → distribución con cola larga (algunos clientes muy antiguos) → imputamos con la **mediana**, más resistente a valores extremos.

In [ ]:
print('Valores faltantes antes de imputar:')
print(df.isna().sum())

df['edad'] = df['edad'].fillna(df['edad'].mean())
df['antiguedad_meses'] = df['antiguedad_meses'].fillna(df['antiguedad_meses'].median())

print('\nValores faltantes después de imputar:')
print(df.isna().sum())

## Sección 3 — Aprendizaje supervisado: CLASIFICACIÓN
### ¿Este cliente va a cancelar el próximo mes?

Esto es un problema de **clasificación**: la variable a predecir (`cancelo`) es una categoría (0 o 1), no un número continuo. Usaremos un modelo de **Regresión Logística**, uno de los algoritmos clásicos más usados en la industria para este tipo de problema (riesgo crediticio, fuga de clientes, diagnóstico binario, detección de fraude, etc.).

**Aplicamos exactamente el flujo de la Sesión 30:**
1. Dividimos en entrenamiento y prueba (nunca evaluamos con los mismos datos con los que entrenamos).
2. Entrenamos el modelo.
3. Evaluamos con la matriz de confusión, precisión, recall y F1 — no solo el accuracy.

In [ ]:
features_clasificacion = ['edad', 'antiguedad_meses', 'uso_mensual', 'quejas_soporte']
X = df[features_clasificacion]
y = df['cancelo']

# División 70% entrenamiento / 30% prueba, estratificada para conservar la proporción de cancelaciones
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Escalamos las variables: buena práctica para modelos lineales (todas las variables en una escala comparable)
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

modelo_clasificacion = LogisticRegression(class_weight='balanced')
modelo_clasificacion.fit(X_train_esc, y_train)

y_pred = modelo_clasificacion.predict(X_test_esc)

print('Modelo entrenado. Estas son sus predicciones sobre clientes que NUNCA vio en el entrenamiento.')

In [ ]:
# Matriz de confusión: la matriz de métricas que permiten ver qué tan bien aprendió el modelo.
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Se queda', 'Cancela'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — Predicción de cancelación')
plt.show()

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy (exactitud global):  {acc:.1%}')
print(f'Precisión (de los que marqué como "cancela", cuántos cancelaron de verdad): {prec:.1%}')
print(f'Recall (de los que cancelaron de verdad, cuántos detecté):                  {rec:.1%}')
print(f'F1-score (equilibrio entre ambas):                                          {f1:.1%}')

**Lectura de negocio:** si el costo de "perder un cliente sin avisar" es alto (por ejemplo, en un seguro o una suscripción B2B), el **recall** es la métrica que más te debe importar: quieres detectar a la mayor cantidad posible de clientes en riesgo, aunque eso signifique contactar a algunos que en realidad no iban a cancelar (falsos positivos).

Es decir, el modelo en este caso, ha sido configurado para ser **'alarmista'**: prioriza activamente no dejar escapar clientes en riesgo (logrando un recall del 52%), incluso a costa de generar una cantidad considerable de falsas alarmas. Una precisión de apenas 39.4% significa que, de cada 10 clientes que el modelo predice que cancelarán, solo 4 realmente lo harán, mientras que los otros 6 son contactados innecesariamente. Esta priorización es un efecto directo de haber usado `class_weight='balanced'`, lo que le indicó al modelo que el costo de un cliente perdido (un *falso negativo*) es significativamente mayor que el costo de contactar de más a alguien que no iba a cancelar (un *falso positivo*).

## Sección 4 — Aprendizaje supervisado: REGRESIÓN
### ¿Cuánto va a gastar este cliente el próximo mes?

Ahora la variable a predecir (`gasto_mensual`) es un **número continuo**, no una categoría. Usamos **Regresión Lineal**, el algoritmo base para predecir ventas, demanda, costos, tiempos de espera, o cualquier cifra continua de negocio.

In [ ]:
features_regresion = ['antiguedad_meses', 'uso_mensual']
Xr = df[features_regresion]
yr = df['gasto_mensual']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.30, random_state=42)

modelo_regresion = LinearRegression()
modelo_regresion.fit(Xr_train, yr_train)

yr_pred = modelo_regresion.predict(Xr_test)

mse = mean_squared_error(yr_test, yr_pred)
rmse = np.sqrt(mse)
r2 = r2_score(yr_test, yr_pred)

print(f'Error Cuadrático Medio (MSE):         {mse:,.1f}')
print(f'Raíz del Error Cuadrático (RMSE):     {rmse:,.1f}  (mismas unidades que el gasto, en miles de CLP)')
print(f'R-cuadrado:                           {r2:.2f}  (proporción de la variación del gasto que el modelo explica)')

plt.scatter(yr_test, yr_pred, alpha=0.5, color='#2E5FA3')
lims = [yr_test.min(), yr_test.max()]
plt.plot(lims, lims, 'r--', label='Predicción perfecta')
plt.xlabel('Gasto real')
plt.ylabel('Gasto predicho')
plt.title('Predicción vs. realidad — Regresión de gasto mensual')
plt.legend()
plt.show()

**Lectura de negocio:** un RMSE bajo (comparado con el gasto promedio) significa que el modelo te sirve para presupuestar con confianza. Un R² cercano a 1 significa que las variables elegidas explican bien el comportamiento; un R² bajo es una señal de que faltan variables relevantes — exactamente la pregunta 1 de la Sesión 30 ("¿qué variable es más difícil de medir?").

### **¿Qué "regla" descubrió el modelo?**

El modelo de regresión nunca vio la fórmula que usamos para generar `gasto_mensual` en la Sección 1. Solo vio pares de datos (antigüedad, uso, gasto) y ajustó una línea que los explicara lo mejor posible. Comparemos lo que **aprendió solo** contra la regla que **nosotros escondimos**:

In [ ]:
# La regla real que usamos para generar los datos en la Sección 1 (normalmente NUNCA la conoceríamos):
coef_reales = {'antiguedad_meses': 0.9, 'uso_mensual': 3.2}
intercepto_real = 25

# Lo que el modelo aprendió mirando solo los datos, sin conocer la fórmula:
coef_aprendidos = dict(zip(features_regresion, modelo_regresion.coef_))

tabla_comparacion = pd.DataFrame({
    'Coeficiente real (oculto)': pd.Series(coef_reales),
    'Coeficiente aprendido por el modelo': pd.Series(coef_aprendidos)
})
print(f'Intercepto real (oculto):      {intercepto_real}')
print(f'Intercepto aprendido:          {modelo_regresion.intercept_:.2f}')
print()
print(tabla_comparacion.round(3))

## Sección 5 — Aprendizaje NO supervisado: Clustering con K-Means
### Descubrir segmentos de clientes sin decirle al modelo qué buscar

A diferencia de las secciones anteriores, aquí **no usamos la variable `cancelo`**. Le pedimos al algoritmo que encuentre grupos naturales de clientes solo mirando su comportamiento (uso y antigüedad). Esto sirve para segmentación de clientes, tipologías de pacientes, perfiles de empleados, categorías de productos, etc., sin necesidad de etiquetas previas.

In [ ]:
features_cluster = ['antiguedad_meses', 'uso_mensual', 'gasto_mensual']
Xc = df[features_cluster]

Xc_esc = StandardScaler().fit_transform(Xc)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['segmento'] = kmeans.fit_predict(Xc_esc)

resumen_segmentos = df.groupby('segmento')[['antiguedad_meses', 'uso_mensual', 'gasto_mensual', 'cancelo']].mean().round(2)
resumen_segmentos['n_clientes'] = df.groupby('segmento').size()
print('Perfil promedio de cada segmento descubierto por K-Means:')
resumen_segmentos

> **Nota:** vamos a agrupar usando **3 variables** (antigüedad, uso y gasto mensual), pero este gráfico solo puede mostrar 2 ejes. El gasto mensual —la variable que más distingue al Segmento 2 de los demás— queda invisible aquí. Por eso dos segmentos pueden verse mezclados en este plano aunque K-Means los haya separado correctamente mirando las 3 variables a la vez. La Sección 6 (PCA) resuelve justamente este problema, resumiendo las 3 variables en un solo plano 2D.

In [ ]:
colores_segmento = {0: '#4B0082', 1: '#1F9E89', 2: '#FDE725'}  # colores discretos, uno por segmento

for seg in sorted(df['segmento'].unique()):
    subset = df[df['segmento'] == seg]
    plt.scatter(subset['antiguedad_meses'], subset['uso_mensual'],
                color=colores_segmento[seg], alpha=0.6, label=f'Segmento {seg}')

plt.xlabel('Antigüedad (meses)')
plt.ylabel('Uso mensual')
plt.title('Segmentos de clientes descubiertos por K-Means (sin usar la etiqueta "cancelo")')
plt.legend(title='Segmento')
plt.show()


**Lectura de negocio:** revisa la columna `cancelo` en la tabla de perfiles. Aunque el modelo nunca vio esa variable, es común que uno de los segmentos concentre una tasa de cancelación mucho más alta que los demás — el clustering "encontró" un patrón de riesgo sin que nadie se lo enseñara explícitamente.

## Sección 6 — Aprendizaje NO supervisado: PCA (reducción de dimensionalidad)
### Simplificar muchas variables correlacionadas en un resumen visual

PCA comprime varias variables numéricas en unos pocos "componentes" que capturan la mayor parte de la variación de los datos. Es muy útil cuando tienes docenas de indicadores (sensores, preguntas de encuesta, variables financieras) y necesitas visualizarlos o simplificarlos sin perder la esencia.

In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(Xc_esc)

df['pca_1'] = componentes[:, 0]
df['pca_2'] = componentes[:, 1]

varianza_explicada = pca.explained_variance_ratio_
print(f'El componente 1 explica {varianza_explicada[0]:.1%} de la variación de los datos.')
print(f'El componente 2 explica {varianza_explicada[1]:.1%} de la variación de los datos.')
print(f'Entre los dos, resumen el {varianza_explicada.sum():.1%} de la información original de 3 variables en solo 2.')

for seg in sorted(df['segmento'].unique()):
    subset = df[df['segmento'] == seg]
    plt.scatter(subset['pca_1'], subset['pca_2'],
                color=colores_segmento[seg], alpha=0.6, label=f'Segmento {seg}')

plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.title('Los mismos clientes y segmentos, vistos en 2 dimensiones (PCA)')
plt.legend(title='Segmento')
plt.show()

## Sección 7 — El dilema del ajuste: subajuste, ajuste óptimo y sobreajuste

Reproducimos en código la curva que vimos en la Sesión 30: un modelo demasiado simple **no aprende** el patrón (subajuste); un modelo demasiado complejo **memoriza el ruido** en vez del patrón (sobreajuste); en el medio está el punto ideal.

In [ ]:
# Generamos una relación no lineal simple entre "inversión en marketing" y "ventas", con ruido
x_dato = np.linspace(0, 10, 60)
y_dato = 3 + 2.5 * x_dato - 0.18 * x_dato**2 + np.random.normal(0, 1.6, 60)

x_train_p, x_test_p, y_train_p, y_test_p = train_test_split(
    x_dato.reshape(-1, 1), y_dato, test_size=0.3, random_state=7
)

grados = {'Subajuste (grado 1)': 1, 'Ajuste óptimo (grado 2)': 2, 'Sobreajuste (grado 15)': 15}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x_linea = np.linspace(0, 10, 200).reshape(-1, 1)

for ax, (titulo, grado) in zip(axes, grados.items()):
    modelo_poly = make_pipeline(PolynomialFeatures(grado), LinearRegression())
    modelo_poly.fit(x_train_p, y_train_p)

    r2_train = r2_score(y_train_p, modelo_poly.predict(x_train_p))
    r2_test = r2_score(y_test_p, modelo_poly.predict(x_test_p))

    ax.scatter(x_train_p, y_train_p, color='#2E5FA3', s=20, label='Entrenamiento')
    ax.scatter(x_test_p, y_test_p, color='#E8792F', s=20, label='Prueba')
    ax.plot(x_linea, modelo_poly.predict(x_linea), color='black', linewidth=1.5)
    ax.set_title(f'{titulo}\nR² train={r2_train:.2f} | R² test={r2_test:.2f}')
    ax.set_ylim(y_dato.min() - 5, y_dato.max() + 5)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Lectura de negocio:** fíjate cómo el modelo de "sobreajuste" tiene un R² de entrenamiento altísimo pero un R² de prueba mucho peor — es la misma "brecha de 37 puntos" que vimos en la Sesión 30, ahora generada por ti mismo. Un modelo así predice muy bien el pasado, pero falla al proyectar el futuro: exactamente lo que ocurrió con los modelos entrenados antes del COVID.

## Sección 8 — Regularización: Ridge vs. Lasso en acción

Comparamos, sobre el mismo dataset de gasto mensual (ahora con más variables, algunas poco útiles a propósito), tres modelos: Regresión Lineal simple, Ridge (L2) y Lasso (L1).

In [ ]:
# Agregamos variables "ruido" que no deberían influir realmente en el gasto, para ver qué hace cada modelo con ellas
df['variable_ruido_1'] = np.random.normal(0, 1, n)
df['variable_ruido_2'] = np.random.normal(0, 1, n)

features_reg_amplio = ['antiguedad_meses', 'uso_mensual', 'edad', 'quejas_soporte',
                         'variable_ruido_1', 'variable_ruido_2']

Xa = StandardScaler().fit_transform(df[features_reg_amplio])
ya = df['gasto_mensual']

Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.3, random_state=42)

modelos = {
    'Regresión Lineal': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=5.0),
    'Lasso (L1)': Lasso(alpha=0.5)
}

tabla_coef = pd.DataFrame(index=features_reg_amplio)

for nombre, modelo in modelos.items():
    modelo.fit(Xa_train, ya_train)
    tabla_coef[nombre] = modelo.coef_.round(2)

print('Comparación de coeficientes (importancia de cada variable para cada modelo):')
tabla_coef

**Lectura de negocio:** observa cómo **Lasso** lleva a (o cerca de) cero los coeficientes de `variable_ruido_1` y `variable_ruido_2` — literalmente las está "eliminando" del modelo — mientras que **Ridge** los reduce pero rara vez a cero. La Regresión Lineal simple, en cambio, les puede asignar peso aunque no tengan relación real con el gasto. Esto es exactamente lo que vimos en la Sesión 30: Lasso para "limpiar" variables sospechosas, Ridge para estabilizar cuando sospechas que todas aportan algo.

## Sección 9 — La trampa del accuracy, revisitada con datos desbalanceados

Construimos un escenario deliberadamente desbalanceado (como el fraude, una enfermedad rara, o una falla de máquina) para ver en vivo por qué el accuracy solo no basta.

In [ ]:
n_desb = 1000
riesgo_alto = np.zeros(n_desb, dtype=int)
riesgo_alto[:50] = 1  # solo 5% de casos de riesgo real
np.random.shuffle(riesgo_alto)

# Un modelo "perezoso" que siempre predice "sin riesgo"
prediccion_perezosa = np.zeros(n_desb, dtype=int)

acc_perezoso = accuracy_score(riesgo_alto, prediccion_perezosa)
rec_perezoso = recall_score(riesgo_alto, prediccion_perezosa, zero_division=0)
f1_perezoso = f1_score(riesgo_alto, prediccion_perezosa, zero_division=0)

print('Modelo "perezoso" que siempre predice "sin riesgo":')
print(f'  Accuracy: {acc_perezoso:.1%}   <- ¡parece excelente!')
print(f'  Recall:   {rec_perezoso:.1%}   <- no detecta NINGÚN caso de riesgo real')
print(f'  F1-score: {f1_perezoso:.1%}')

**Lectura de negocio:** un 95% de accuracy suena a un gran modelo, pero el recall de 0% revela que es inútil para su propósito real. Esta es la razón por la que, en la Sesión 30, insistimos en nunca reportar solo el accuracy cuando las clases están desbalanceadas — algo muy común en fraude, fallas de equipos, enfermedades raras o defectos de calidad.

## Sección 10 — Síntesis: la caja de herramientas del arquitecto

Antes de elegir un algoritmo, hazte estas preguntas en orden:

In [ ]:
def recomendar_enfoque(tengo_etiquetas, quiero_descubrir_patrones, entorno_dinamico, problema_muy_complejo):
    """Árbol de decisión simplificado del Capítulo 4 / Sesión 31.
    Responde True/False a cada pregunta y obtén una recomendación."""
    if problema_muy_complejo:
        return 'Aprendizaje Profundo (redes neuronales) — imágenes, audio, texto, gran volumen de datos.'
    if entorno_dinamico:
        return 'Aprendizaje por Refuerzo — agentes que aprenden por ensayo y error maximizando una recompensa.'
    if tengo_etiquetas:
        return 'Aprendizaje Supervisado — Regresión (predicción numérica) o Clasificación (predicción de categoría).'
    if quiero_descubrir_patrones:
        return 'Aprendizaje No Supervisado — K-Means (segmentar) o PCA (simplificar/visualizar).'
    return 'Vuelve a evaluar tus datos: probablemente necesitas más contexto sobre qué problema quieres resolver.'

# Ejemplo: "¿este cliente cancelará?" -> sí tengo etiquetas (cancelo: sí/no), no es un entorno dinámico, no es súper complejo
print(recomendar_enfoque(tengo_etiquetas=True, quiero_descubrir_patrones=False,
                          entorno_dinamico=False, problema_muy_complejo=False))

# Ahora prueba tú: cambia estos valores por el caso de "regla fija" que compartiste al inicio de la clase
print(recomendar_enfoque(tengo_etiquetas=False, quiero_descubrir_patrones=True,
                          entorno_dinamico=False, problema_muy_complejo=False))

### Reto final de cierre

En parejas (5 minutos):
1. Vuelve a la "regla fija" que tu empresa usa hoy (la que compartiste en la apertura de esta sesión o de la Sesión 30).
2. Usa la función `recomendar_enfoque()` de arriba para clasificarla.
3. Si te recomienda Supervisado: ¿sería un problema de clasificación o de regresión? ¿Qué variable sería tu objetivo (`y`)?
4. Si te recomienda No Supervisado: ¿qué variables usarías para segmentar?

**Próxima sesión (32):** cuando el problema es demasiado complejo para estos algoritmos clásicos —imágenes, audio, lenguaje natural—, entramos al territorio de las redes neuronales y el Aprendizaje Profundo.